# UrbanHub, Partie 4 : analyse croisée météo × pollution × mobilité

Les trois flux sont joints sur **(ville, heure UTC)** dans la vue `analytics.v_city_hourly_all` :

| Flux | Source | Couverture utilisée ici |
|---|---|---|
| Météo | NOAA (jusqu'au 24/08/2025) puis Open-Meteo | horaire, 11 villes, 2022 → aujourd'hui |
| Pollution | OpenAQ (capteurs fixes, µg/m³) | horaire, 11 villes, 31/08/2026 → aujourd'hui |
| Vélos | archives CityBikes (changements d'état des stations) + flux temps réel | horaire, 10 villes, septembre 2026 complet |

**Indicateur d'usage des vélos** : `departures_per_station` = vélos retirés dans l'heure par station
(variations du stock, hors camions de rééquilibrage). C'est une mesure d'usage réel, et non de stock.
Comme l'usage dépend fortement de l'heure et du jour, on compare chaque heure à **l'usage attendu**
(moyenne de la même ville, même heure, semaine ou week-end) : `usage_index = observé / attendu`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf
from urbanhub_common import read_sql

pd.set_option("display.float_format", "{:.2f}".format)
POLL = ["pm25", "pm10", "no2", "o3"]
METEO = ["temperature", "wind_speed", "precipitation", "pressure", "visibility"]
COLORS = {"pm25": "#2a78d6", "pm10": "#eb6834", "no2": "#1baf7a", "o3": "#eda100"}

df = read_sql("SELECT * FROM analytics.v_city_hourly_all WHERE hour_utc >= '2026-08-31'")
df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True)
df["rain"] = (df["precipitation"] > 0.1)
print(f"{len(df):,} heures × ville | pollution : {df['no2'].notna().sum():,} | vélos (usage) : "
      f"{df['departures'].notna().sum():,} | les 3 flux : {(df['departures'].notna() & df['no2'].notna()).sum():,}")

## Q1. Existe-t-il une relation entre conditions météorologiques et pollution ?

In [ ]:
d1 = df.dropna(subset=["no2", "temperature"]).copy()
raw = d1[POLL + METEO].corr(method="spearman").loc[POLL, METEO]

# même calcul sur les ANOMALIES : on retire le cycle horaire / semaine propre à chaque ville,
# sinon on mesure surtout "il fait plus chaud l'après-midi et il y a plus d'O3 l'après-midi"
grp = ["city", "hour_local", "is_weekend"]
anom = d1[POLL + METEO] - d1.groupby(grp)[POLL + METEO].transform("mean")
ano = anom.corr(method="spearman").loc[POLL, METEO]

print("Corrélation de Spearman (brute)"); display(raw.round(2))
print("Corrélation de Spearman (anomalies, cycle journalier retiré)"); display(ano.round(2))
px.imshow(ano.round(2), text_auto=True, zmin=-0.6, zmax=0.6, aspect="auto",
          color_continuous_scale=["#2a78d6", "#f0efec", "#e34948"],
          title="Pollution × météo : corrélation des anomalies (Spearman)").show()

> 📖 **Comment lire ce graphique.** Deux tableaux de corrélation de Spearman (de -1 à +1) entre polluants (lignes) et variables météo (colonnes). Le premier est **brut**. Le second porte sur les **anomalies** : on retire d'abord la moyenne de la même ville, de la même heure et du même type de jour, ce qui supprime la fausse corrélation due au cycle journalier (l'après-midi est à la fois plus chaud et plus riche en ozone). C'est le second qu'il faut interpréter. Bleu = relation négative, rouge = positive.

In [ ]:
# Effet du vent et de la pluie : concentration moyenne par classe
d1["vent"] = pd.cut(d1["wind_speed"], [0, 5, 10, 15, 20, 30, 100],
                    labels=["0-5", "5-10", "10-15", "15-20", "20-30", ">30"], include_lowest=True)
vent = d1.groupby("vent", observed=True)[["no2", "pm25", "pm10", "o3"]].mean()
display(vent.round(1))
long = vent.reset_index().melt("vent", var_name="polluant", value_name="µg/m³")
px.line(long, x="vent", y="µg/m³", color="polluant", markers=True, color_discrete_map=COLORS,
        title="Concentration moyenne selon la vitesse du vent (km/h)").show()

d1["temp"] = pd.cut(d1["temperature"], [-10, 5, 10, 15, 20, 25, 40])
display(d1.groupby("temp", observed=True)[["o3", "no2"]].mean().round(1).T)
display(d1.groupby("rain")[["no2", "pm25", "pm10", "o3"]].mean().round(1)
        .rename(index={False: "sec", True: "pluie > 0,1 mm"}))

> 📖 **Comment lire ce graphique.** Concentration moyenne par classe de vent (courbes), par classe de température (tableau O3/NO2) et par temps sec ou pluvieux. Une courbe qui descend signifie que le vent dilue le polluant.

In [ ]:
# Modèle multivarié : effet propre de chaque variable météo, à heure / jour / ville égaux
rows = []
for p in POLL:
    m = smf.ols(f"np.log({p} + 1) ~ wind_speed + temperature + rain + pressure"
                " + C(hour_local) + is_weekend + C(city)", data=d1.dropna(subset=[p, "pressure"])).fit()
    for var, step in [("wind_speed", 10), ("temperature", 5), ("rain[T.True]", 1), ("pressure", 10)]:
        rows.append({"polluant": p, "variable": var.replace("[T.True]", ""), "pas": step,
                     "effet_%": (np.exp(m.params[var] * step) - 1) * 100, "p_value": m.pvalues[var],
                     "R2": m.rsquared})
q1_model = pd.DataFrame(rows)
q1_model.pivot(index="variable", columns="polluant", values="effet_%").round(0)

> 📖 **Comment lire ce graphique.** Résultat d'une **régression linéaire** sur le logarithme de la concentration, avec l'heure, le jour et la ville en variables de contrôle. Chaque case donne l'effet en % d'un changement donné, à conditions égales par ailleurs : +10 km/h de vent, +5 °C, présence de pluie, +10 hPa. Exemple : -39 pour le NO2 et le vent signifie 39 % de NO2 en moins quand le vent augmente de 10 km/h.

### Interprétation Q1 : oui, la météo explique une grande part de la pollution

**Le vent est le premier facteur.** À heure, jour et ville égaux, +10 km/h de vent fait baisser le **NO2 de 39 %**, les **PM2.5 de 25 %** et les **PM10 de 19 %** (p < 0,001). Par temps calme (0-5 km/h), le NO2 moyen est de 25,3 µg/m³ ; au-delà de 30 km/h, il tombe à 9,1 µg/m³, soit 2,8 fois moins. Même une fois le cycle journalier retiré, la corrélation NO2 × vent vaut **-0,51**. Le vent disperse les polluants émis localement (trafic, chauffage).

**L'ozone fait l'inverse.** Il augmente avec la **température** : +31 % par +5 °C, de 17 µg/m³ sous 5 °C à 91 µg/m³ au-dessus de 25 °C (corrélation 0,49). C'est un polluant **secondaire**, formé au soleil à partir du NO2. Il augmente aussi avec le vent (+37 % par +10 km/h) : quand le NO2 est dispersé, il ne détruit plus l'ozone localement, et le vent apporte un air de fond plus riche en O3.

**La pluie a un effet de lessivage modéré** : NO2 -10 %, PM10 -11 %. Les PM2.5 augmentent légèrement (+9 %), probablement à cause de l'humidité, qui fait grossir les particules fines.

Météo + heure + ville expliquent **51 % de la variance du NO2** et **53 % de celle de l'O3** (R²), contre 23 à 28 % pour les particules, davantage influencées par des sources non météorologiques.

**À retenir** : les situations à risque sont l'**air stagnant** (vent faible) pour le NO2 et les particules, et la **chaleur ensoleillée** pour l'ozone.

## Q2. Les conditions météo influencent-elles l'utilisation des vélos urbains ?

On utilise le mois de septembre 2026 (archives CityBikes), entre 7 h et 22 h (heure locale).

In [ ]:
d2 = df.dropna(subset=["departures_per_station", "temperature"]).copy()
d2["attendu"] = d2.groupby(["city", "hour_local", "is_weekend"])["departures_per_station"].transform("mean")
d2 = d2[(d2["hour_local"].between(7, 22)) & (d2["attendu"] > 0.05)]
d2["usage_index"] = d2["departures_per_station"] / d2["attendu"]

d2["pluie"] = pd.cut(d2["precipitation"], [-0.01, 0.1, 1, 100], labels=["sec", "pluie faible (0,1-1 mm)", "pluie forte (> 1 mm)"])
d2["temp"] = pd.cut(d2["temperature"], [-10, 10, 14, 18, 22, 26, 40], labels=["<10", "10-14", "14-18", "18-22", "22-26", ">26"])
d2["vent"] = pd.cut(d2["wind_speed"], [0, 10, 20, 30, 100], labels=["0-10", "10-20", "20-30", ">30"], include_lowest=True)

for col in ["pluie", "temp", "vent"]:
    t = d2.groupby(col, observed=True)["usage_index"].agg(["mean", "count"])
    t["écart vs attendu"] = (t["mean"] - 1) * 100
    display(t.rename(columns={"mean": "usage_index", "count": "heures"}).round(2))

> 📖 **Comment lire ce graphique.** Pour chaque classe de pluie, de température et de vent : `usage_index` moyen (1 = usage habituel) et écart en % par rapport à l'attendu. La colonne `heures` indique combien d'heures fondent la moyenne : en dessous d'environ 30 heures, le chiffre est peu fiable.

In [ ]:
fig = px.box(d2, x="pluie", y="usage_index", points=False, color_discrete_sequence=["#2a78d6"],
             title="Usage des vélos (observé / attendu) selon la pluie", height=380)
fig.add_hline(y=1, line_dash="dot")
fig.update_yaxes(range=[0, 2])
fig.show()

m2 = smf.ols("np.log(usage_index + 0.05) ~ precipitation + temperature + I(temperature**2) + wind_speed + C(city)",
             data=d2).fit()
display(m2.summary().tables[1])
eff = {"pluie +1 mm/h": m2.params["precipitation"], "vent +10 km/h": m2.params["wind_speed"] * 10}
print({k: f"{(np.exp(v) - 1) * 100:+.0f} %" for k, v in eff.items()})

> 📖 **Comment lire ce graphique.** Boîte à moustaches de l'usage selon la pluie (ligne pointillée = usage habituel), puis régression : effet en % de +1 mm/h de pluie et de +10 km/h de vent sur l'usage.

In [ ]:
par_ville = (d2.groupby(["city", "pluie"], observed=True)["usage_index"].mean().unstack()
             .assign(effet_pluie_pct=lambda t: (t.iloc[:, -1] / t["sec"] - 1) * 100))
par_ville.round(2)

> 📖 **Comment lire ce graphique.** Usage (observé / attendu) par ville selon la pluie, et effet de la pluie en %. Une case vide signifie qu'il n'a pas plu pendant les heures disponibles de cette ville.

### Interprétation Q2 : oui, surtout la pluie

**La pluie est le seul effet net.** Quand il pleut, on retire **environ 22 % de vélos en moins** qu'à une heure comparable (même ville, même heure, même type de jour) : -22 % sous pluie faible, -21 % sous pluie forte. Le modèle donne -7 % par mm/h (p < 0,001). L'effet est marqué à **Strasbourg (-37 %)**, **Paris (-31 %)**, Lille (-21 %) et Rennes (-20 %).

**Le vent pèse un peu** : -6 % d'usage entre 20 et 30 km/h, -5 % par +10 km/h dans le modèle (p < 0,001).

**La température n'a pas d'effet mesurable** dans la plage observée en septembre (10-30 °C, p = 0,99). Seules les 11 heures sous 10 °C montrent -14 %, ce qui est trop peu pour conclure.

Le R² très faible (0,025) est en soi un résultat : **l'usage du vélo suit d'abord la routine** (trajets domicile-travail, pics à 8 h et 18 h). La météo ne fait que moduler cette routine, et seule la pluie la modifie nettement.

*Limites* : un seul mois (septembre 2026), sec à 98 % (87 heures de pluie seulement). L'archive a des pannes (Lyon, Nantes et Toulouse du 1er au 10/09, quelques heures à Paris), exclues de l'analyse. L'indicateur compte des variations de stock : il sous-estime les trajets lorsque deux trajets se compensent entre deux relevés.

## Q3. Peut-on identifier des conditions météorologiques favorables à la mobilité douce ?

1. On lit, dans les données vélo, la combinaison température × pluie où l'usage dépasse l'attendu.
2. On en tire une **règle de confort**, qu'on applique ensuite à **4 ans et demi d'historique météo**
   (2022-2026) pour savoir quand et où ces conditions sont réunies.

In [ ]:
heat = d2.pivot_table(index="temp", columns="pluie", values="usage_index", aggfunc="mean", observed=True)
px.imshow(heat.round(2), text_auto=True, zmin=0.6, zmax=1.4, aspect="auto",
          color_continuous_scale=["#e34948", "#f0efec", "#1baf7a"],
          title="Usage des vélos (observé / attendu) : température × pluie").show()
heat.round(2)

> 📖 **Comment lire ce graphique.** Carte de chaleur température × pluie : vert = plus de vélos que d'habitude, rouge = moins. On y lit les conditions où la mobilité douce se maintient ou progresse.

In [ ]:
# Règle de confort (calibrée sur le tableau ci-dessus) : sec, température douce, vent modéré
T_MIN, T_MAX, VENT_MAX = 12, 28, 25
def favorable(t):
    return (t["precipitation"] <= 0.1) & t["temperature"].between(T_MIN, T_MAX) & (t["wind_speed"] < VENT_MAX)

d2["favorable"] = favorable(d2)
print("Usage moyen (index) - heures favorables :", round(d2.loc[d2.favorable, "usage_index"].mean(), 2),
      "| défavorables :", round(d2.loc[~d2.favorable, "usage_index"].mean(), 2))

hist = read_sql('''SELECT city, hour_utc, temperature, wind_speed, precipitation
                   FROM analytics.v_meteo_city_hourly''')
hist["hour_utc"] = pd.to_datetime(hist["hour_utc"], utc=True)
loc = hist["hour_utc"].dt.tz_convert("Europe/Paris")
hist = hist[loc.dt.hour.between(7, 21)].assign(mois=loc.dt.month, annee=loc.dt.year)
hist["favorable"] = favorable(hist)
mois = hist.groupby(["city", "mois"])["favorable"].mean().unstack()
px.imshow((mois * 100).round(0), text_auto=True, aspect="auto", color_continuous_scale="Greens",
          labels={"color": "% heures"}, title="Part des heures de journée (7 h-21 h) favorables au vélo, 2022-2026").show()
print("Classement des villes (part annuelle d'heures favorables) :")
display((hist.groupby("city")["favorable"].mean() * 100).sort_values(ascending=False).round(1))

> 📖 **Comment lire ce graphique.** La règle de confort est appliquée à toute la météo horaire 2022-2026, de 7 h à 21 h. Chaque case donne la part des heures favorables pour une ville et un mois. Le classement final donne la part sur l'année.

In [ ]:
# Bonus qualité de l'air : heures favorables ET air sain (sous les seuils OMS), sur la période avec pollution
d3 = df.dropna(subset=["temperature", "pm25", "no2"]).copy()
d3 = d3[d3["hour_local"].between(7, 21)]
d3["meteo_ok"] = favorable(d3)
d3["air_ok"] = (d3["pm25"] < 15) & (d3["no2"] < 25) & (d3["o3"].fillna(0) < 100)
t = d3.groupby("city")[["meteo_ok", "air_ok"]].mean()
t["meteo_et_air_ok"] = d3.groupby("city").apply(lambda g: (g.meteo_ok & g.air_ok).mean(), include_groups=False)
(t * 100).round(0).sort_values("meteo_et_air_ok", ascending=False)

> 📖 **Comment lire ce graphique.** Part des heures où la météo est favorable (`meteo_ok`), où l'air est sain (`air_ok` : PM2.5 < 15, NO2 < 25, O3 < 100 µg/m³), et où les deux sont réunis, sur la période où la pollution est disponible.

### Interprétation Q3 : oui. Temps sec, 12-28 °C, vent < 25 km/h

Le tableau température × pluie le montre : **par temps sec, l'usage reste au niveau attendu entre 10 et 30 °C** (index 0,98-1,02), tandis que **la pluie le fait tomber à 0,72-0,82**. La condition décisive est donc l'absence de pluie, complétée par une température douce et un vent modéré. D'où la règle : **sec, 12-28 °C, vent < 25 km/h**.

Appliquée à **4,5 ans d'historique (2022-2026, de 7 h à 21 h)** :
- **Les meilleurs mois sont septembre (73 % d'heures favorables), octobre (70 %) et mai (67 %).** Juillet-août (50-53 %) est pénalisé par la chaleur au-delà de 28 °C, surtout dans le Sud : en juillet, Marseille n'a que 23 % d'heures favorables, contre 67 % à Paris et Rennes. L'hiver est quasi défavorable (8 % en janvier).
- **Villes** : Nice (43 % des heures de l'année), Montpellier (39 %) et Bordeaux (37 %) sont les plus favorables. Lille (29 %), Strasbourg (30 %) et Lyon (30 %) le sont le moins.

Si l'on ajoute la **qualité de l'air** (PM2.5 < 15, NO2 < 25, O3 < 100 µg/m³, de septembre à octobre 2026), le classement change. **Nantes (81 % d'heures « météo et air OK »)**, Rennes (73 %) et Bordeaux (70 %) offrent le meilleur compromis. **Marseille tombe à 28 %** : la météo y est favorable 84 % du temps, mais l'air n'y est sain que 36 % du temps (ozone et NO2). Paris est à 56 %.

## Q4. Existe-t-il des périodes où mobilité, pollution et météo interagissent fortement ?

On passe à l'échelle **ville × jour** (septembre 2026, les 3 flux disponibles) :
- corrélations entre les indicateurs journaliers ;
- détection des **jours d'interaction forte**, où au moins 3 indicateurs s'écartent nettement (|z| > 1) de la normale
  de la ville, en même temps.

In [ ]:
day = (df.dropna(subset=["departures"]).assign(date=lambda t: t["date_local"])
       .groupby(["city", "date"])
       .agg(temperature=("temperature", "mean"), pluie_mm=("precipitation", "sum"), vent=("wind_speed", "mean"),
            retraits=("departures", "sum"), heures_velo=("departures", "count"), no2=("no2", "mean"),
            pm25=("pm25", "mean"), o3=("o3", "mean"), weekend=("is_weekend", "first"))
       .reset_index())
day = day[day["heures_velo"] >= 20]   # jours complets seulement (pannes de l'archive, jours coupés en fin de mois)
# usage corrigé du jour de la semaine : retraits / moyenne de la ville pour le même type de jour
day["usage_index"] = day["retraits"] / day.groupby(["city", "weekend"])["retraits"].transform("mean")
day = day.dropna(subset=["no2"])
cols = ["temperature", "pluie_mm", "vent", "usage_index", "no2", "pm25", "o3"]
cd = day[cols].corr(method="spearman")
px.imshow(cd.round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale=["#2a78d6", "#f0efec", "#e34948"],
          title="Corrélations journalières ville × jour (Spearman)").show()
print(len(day), "jours × ville avec les 3 flux")

> 📖 **Comment lire ce graphique.** Corrélations entre indicateurs **journaliers** : chaque observation est une ville × un jour. On cherche les couples qui varient ensemble d'un jour à l'autre.

In [ ]:
z = day.groupby("city")[cols].transform(lambda s: (s - s.mean()) / s.std())
z.columns = ["z_" + c for c in cols]
day = pd.concat([day, z], axis=1)
day["n_forts"] = (z.abs() > 1).sum(axis=1)

def profil(r):
    if r.z_pluie_mm > 1 and r.z_usage_index < -1:
        return "jour pluvieux : vélos en baisse" + (", air lavé" if r.z_no2 < 0 and r.z_pm25 < 0 else "")
    if r.z_vent < -1 and (r.z_no2 > 1 or r.z_pm25 > 1):
        return "air stagnant (vent faible) : pollution en hausse" + (", vélos en hausse" if r.z_usage_index > 0.5 else "")
    if r.z_temperature > 1 and r.z_o3 > 1:
        return "chaleur : ozone en hausse" + (", vélos en hausse" if r.z_usage_index > 0.5 else "")
    if r.z_vent > 1 and (r.z_no2 < -1 or r.z_pm25 < -1):
        return "vent fort : air propre" + (", vélos en baisse" if r.z_usage_index < -0.5 else "")
    if r.z_usage_index > 1 and r.pluie_mm < 0.5:
        return "beau temps sec : vélos en hausse" + (", ozone en hausse" if r.z_o3 > 0.5 else "")
    return "autre combinaison"

forts = day[day["n_forts"] >= 3].copy()
forts["profil"] = forts.apply(profil, axis=1)
print(f"{len(forts)} jours d'interaction forte sur {len(day)}")
display(forts["profil"].value_counts())
display(forts.sort_values("n_forts", ascending=False)[
    ["city", "date", "profil", "temperature", "pluie_mm", "vent", "usage_index", "no2", "pm25", "o3"]].head(15).round(1))

> 📖 **Comment lire ce graphique.** Pour chaque ville, chaque indicateur est converti en **z-score** (écart à la moyenne de la ville, en écarts-types). Un jour est d'**interaction forte** si au moins 3 indicateurs ont |z| > 1 en même temps. Un profil résume la combinaison observée.

In [ ]:
# Calendrier : jours forts par ville
px.scatter(forts, x="date", y="city", color="profil", size="n_forts", height=420,
           title="Jours d'interaction forte météo × pollution × vélos (septembre 2026)").show()

# Même logique heure par heure : quand l'usage dévie de l'attendu, la pollution dévie-t-elle aussi ?
h = d2.dropna(subset=["no2"]).copy()
for c in ["no2", "pm25"]:
    h[c + "_anom"] = h[c] - h.groupby(["city", "hour_local", "is_weekend"])[c].transform("mean")
print("Corrélation usage_index × anomalie NO2 :", round(h["usage_index"].corr(h["no2_anom"], method="spearman"), 2),
      "| × anomalie PM2.5 :", round(h["usage_index"].corr(h["pm25_anom"], method="spearman"), 2))

> 📖 **Comment lire ce graphique.** Calendrier des jours d'interaction forte (taille = nombre d'indicateurs extrêmes), puis corrélation heure par heure entre l'usage des vélos et l'anomalie de pollution.

### Interprétation Q4 : oui, sur environ un quart des jours

**65 jours × ville sur 250 (26 %)** montrent au moins 3 indicateurs simultanément hors norme. Les corrélations journalières font apparaître trois mécanismes :

1. **Vent fort, souvent avec pluie : air propre et moins de vélos (18 jours).** Corrélations : vent × NO2 = -0,52, pluie × usage = -0,30, pluie × NO2 = -0,30. Exemple : Lille, 08/09 (14 km/h, 5,4 mm) : NO2 à 6,8 µg/m³ et usage -20 %.
2. **Air stagnant (vent faible) : la pollution monte (10 jours).** Exemple : Nantes, 14/09 (vent 4,9 km/h), NO2 au plus haut du mois.
3. **Chaleur et beau temps sec : ozone en hausse et plus de vélos (19 jours).** Corrélation température × O3 = 0,56. Exemple : Montpellier, 04/09 (26 °C, sec) : O3 à 89 µg/m³ et **usage +50 %**.

**Point clé pour les décideurs** : l'usage du vélo est *positivement* corrélé au NO2 (+0,19 par jour) et aux PM2.5 (+0,14). Ce n'est **pas** le vélo qui pollue : c'est un **facteur commun**, le temps calme, sec et doux, qui attire les cyclistes **et** empêche la dispersion des polluants. **Les jours où l'on pédale le plus sont aussi ceux où les cyclistes sont le plus exposés.** Les alertes pollution (ozone, NO2) devraient donc viser en priorité les usagers des mobilités douces ces jours-là, alors que les jours venteux et pluvieux offrent l'air le plus sain, mais moins de cyclistes.

## Conclusion

| Question | Réponse | Effet principal |
|---|---|---|
| Q1 Météo → pollution | **Oui, fort** | +10 km/h de vent : NO2 -39 %, PM2.5 -25 %. +5 °C : O3 +31 %. R² 0,51-0,53 pour NO2 et O3 |
| Q2 Météo → vélos | **Oui, surtout la pluie** | pluie : -22 % de vélos retirés ; vent : -5 % par +10 km/h ; température : effet nul entre 10 et 30 °C |
| Q3 Conditions favorables | **Sec, 12-28 °C, vent < 25 km/h** | septembre-octobre et mai ; Nice, Montpellier, Bordeaux ; avec la qualité de l'air : Nantes, Rennes, Bordeaux |
| Q4 Interactions fortes | **Oui, 26 % des jours** | vent et pluie : air propre, moins de vélos ; temps calme et chaud : pollution et ozone, plus de vélos |

**Limites**
- Un seul mois de données vélo complètes (septembre 2026, avec des pannes d'archive), et peu d'heures de pluie.
- Météo 2026 issue du modèle Open-Meteo (un point par ville), et non de stations.
- Agrégation à l'échelle de la ville.
- Corrélation n'est pas causalité.
- Seuil OMS de l'O3 appliqué à des valeurs horaires.

**Pour aller plus loin** : charger les archives CityBikes d'autres mois (`cross_analysis archive --months 202605 202606 ...`) et étendre la pollution OpenAQ sur la même période, afin de mesurer l'effet du froid et de la saisonnalité.